> 在线讲义：[第 47 章 · 推理成本模型：roofline 与算术强度、prefill vs decode、KV cache、批处理与延迟](https://wikieden.github.io/math-and-eval-stats/lessons/47-inference-cost.html) · 本书《数学与实验统计》

# 第 47 章 · 推理成本模型：roofline 与算术强度、prefill vs decode、KV cache、批处理与延迟

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/47-inference-cost.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 拐点 | 峰值/带宽 | `ridge()` | 295 |
| 第一步 强度 | 2mkn/… | `matmul_intensity(m, 4096, 4096)` | 1.0 … 1024 |
| 第三步 KV | 2·L·n_kv·d·B | `kv_bytes_per_token(8)` | 128 KiB |
| 第四步 decode | (权重 + KV)/带宽 | `decode_floor_ms(64, 4096)` | 45.0 ms |

例子：一个 7B 级的模型（32 层、hidden 4096、32 个注意力头、每头 128 维，参数约 6.7B，bf16）跑在一张 H100 SXM 上
（bf16 稠密算力约 989 TFLOP/s、显存带宽约 3.35 TB/s）。
里程碑 ME-M29 的“KV cache 计算器（MHA / GQA / MLA）”和“用 roofline 判断任意配置受什么限制”留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
PEAK, BW = 989e12, 3.35e12        # FLOP/s、byte/s
LAYERS, HIDDEN, HEADS, HDIM, PARAMS = 32, 4096, 32, 128, 6.7e9

**函数 `ridge`**：roofline 的拐点：算术强度（FLOP/byte）超过它才是算力受限。

In [ ]:
def ridge() -> float:
    """roofline 的拐点：算术强度（FLOP/byte）超过它才是算力受限。"""
    return PEAK / BW

**函数 `matmul_intensity`**：(m×k)·(k×n)：2mkn 次浮点运算，至少读写 (mk + kn + mn) 个元素。

In [ ]:
def matmul_intensity(m: int, k: int, n: int, bytes_per: int = 2) -> float:
    """(m×k)·(k×n)：2mkn 次浮点运算，至少读写 (mk + kn + mn) 个元素。"""
    return 2 * m * k * n / (bytes_per * (m * k + k * n + m * n))

**函数 `attainable`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def attainable(intensity: float) -> float:
    return min(PEAK, intensity * BW)

**函数 `kv_bytes_per_token`**：每个 token 的 KV cache：K 和 V 各一份，每层 kv_heads × head_dim 个数。

In [ ]:
def kv_bytes_per_token(kv_heads: int, layers: int = LAYERS, hdim: int = HDIM, bytes_per: int = 2) -> int:
    """每个 token 的 KV cache：K 和 V 各一份，每层 kv_heads × head_dim 个数。"""
    return 2 * layers * kv_heads * hdim * bytes_per

**函数 `decode_floor_ms`**：decode 一步的下限（带宽受限）：把权重读一遍 + 把这批请求的 KV cache 读一遍，除以带宽。

In [ ]:
def decode_floor_ms(batch: int, ctx: int, kv_heads: int = HEADS) -> float:
    """decode 一步的下限（带宽受限）：把权重读一遍 + 把这批请求的 KV cache 读一遍，除以带宽。"""
    return (2 * PARAMS + batch * ctx * kv_bytes_per_token(kv_heads)) / BW * 1e3

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 roofline：算术强度决定受算力还是带宽限制")
print(f"  H100 SXM 拐点 {ridge():.0f} FLOP/byte")
for tok in (1, 16, 256, 2048):
    i = matmul_intensity(tok, HIDDEN, HIDDEN)
    print(f"  一次处理 {tok} 个 token 过 4096×4096 的线性层：强度 {i:.1f}，可达 {attainable(i) / 1e12:.0f} TFLOP/s"
          f"（{'算力' if i > ridge() else '带宽'}受限）")

In [ ]:
print("\n第二步 prefill vs decode")
print(f"  prefill 2048 个 token：一次矩阵乘处理整段，强度约 {matmul_intensity(2048, HIDDEN, HIDDEN):.0f}")
print(f"  decode：每步每个请求只有 1 个新 token，batch = 1 时强度约 {matmul_intensity(1, HIDDEN, HIDDEN):.2f}")

In [ ]:
print("\n第三步 KV cache 大小")
for name, kvh in (("MHA（32 个 KV 头）", 32), ("GQA（8 个 KV 头）", 8)):
    per = kv_bytes_per_token(kvh)
    print(f"  {name}：每 token {per / 2**10:.0f} KiB；4096 token 一个请求 {per * 4096 / 2**30:.2f} GiB")

In [ ]:
print("\n第四步 decode 一步的时间下限：读权重 + 读 KV")
for b in (1, 8, 64):
    print(f"  batch {b}、上下文 4096、MHA：{decode_floor_ms(b, 4096):.1f} ms/步，吞吐上限 {b / decode_floor_ms(b, 4096) * 1e3:.0f} token/s")
print(f"  同样 batch 64 换成 GQA：{decode_floor_ms(64, 4096, 8):.1f} ms/步")

## 3. 代码里才看得到的

1. **峰值用的是稠密 bf16 的 989 TFLOP/s**：厂商标的 1979 是 2:4 稀疏的数，实际能跑到的稠密算力还要再打七八折。
2. **decode 下限只算了读**：没算注意力的计算、kernel 启动开销和通信，真实 TPOT 一般是下限的 1.5–3 倍。
3. **只有一组配置**：给任意模型（MHA / GQA / MLA）算 KV、判断 prefill / decode 受什么限制的计算器留给自己做（里程碑 ME-M29）。

## 4. 自己改一改

1. 把 `PARAMS` 换成 70e9、`LAYERS` 换成 80，batch 1 的 decode 下限是多少？
2. 把 KV cache 量化成 int8（每元素 1 字节），batch 64 的 decode 下限变成多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 性能取峰值和强度乘带宽的小者，拐点约三百；
> 线性层强度约等于一起处理的 token 数；
> prefill 卡算力定首字，decode 卡带宽定每字；
> KV 每 token 二乘层乘头乘维乘字节；
> batch 摊薄权重读取，KV 小才开得大。